# 第14章 - 通信与互联网技术（一）：网络协议栈
# Chapter 14 - Communication & Internet Technologies (Part 1): Protocol Stack

---

## 🎯 学习目标 (Learning Objectives)

本节课对应 **Cambridge A-Level Computer Science (9618) Paper 3** 考纲要求：

| 考纲编号 | 内容 |
|---------|------|
| 14.1 | 理解 **TCP/IP protocol stack** 的各层功能 |
| 14.2 | 比较 **OSI model** 与 TCP/IP 模型 |
| 14.3 | 区分 **TCP** 与 **UDP** 协议 |
| 14.4 | 理解 **IP addressing**（IPv4 和 IPv6）、**subnetting** 和 **CIDR** 表示法 |
| 14.5 | 理解 **port numbers** 和 **sockets** 的作用 |

---

> **前置知识**：你需要了解基本的二进制数制和计算机网络的基本概念（如 LAN、WAN）。

In [ ]:
# ============================================================
# 环境配置 - 每次运行 notebook 前都需要先执行此 cell
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import struct
import socket
import ipaddress

# 中文字体配置
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("✅ 环境配置完成！")

---

## 1. 什么是协议栈？(What is a Protocol Stack?)

### 1.1 核心概念

**Protocol Stack（协议栈）** 是一组分层组织的网络协议的集合。每一层负责特定的通信功能，层与层之间通过定义好的接口进行交互。

为什么需要分层？想象寄一封国际信件：

```
你（写信内容）        →  Application Layer（应用层）
    ↓
装入信封、写地址       →  Transport Layer（传输层）
    ↓
邮局选择投递路线       →  Network Layer（网络层）
    ↓
邮车在公路上运输       →  Data Link Layer（数据链路层）
    ↓
公路本身（物理介质）    →  Physical Layer（物理层）
```

**分层的优势**：
- **模块化 (Modularity)**：每层独立开发和更新
- **互操作性 (Interoperability)**：不同厂商的设备可以互通
- **简化设计 (Simplified Design)**：复杂问题分解为小问题
- **易于调试 (Easier Debugging)**：可以逐层排查问题

---

## 2. TCP/IP 协议栈 (TCP/IP Protocol Stack)

**TCP/IP** 是互联网实际使用的协议栈，共分为 **4层**（有些教材将其扩展为5层以便与 OSI 对比）。

### 2.1 五层模型详解

| 层次 | 英文名 | 主要功能 | 典型协议/技术 | 数据单位 |
|------|--------|---------|--------------|----------|
| **5. 应用层** | Application Layer | 为用户应用程序提供网络服务 | HTTP, FTP, SMTP, DNS, SSH | Data (数据) |
| **4. 传输层** | Transport Layer | 端到端的可靠传输或快速传输 | **TCP**, **UDP** | Segment (段) |
| **3. 网络层** | Network Layer | 逻辑寻址和路由选择 | **IP**, ICMP, ARP | Packet (包) |
| **2. 数据链路层** | Data Link Layer | 物理寻址、帧同步、差错控制 | Ethernet, Wi-Fi, PPP | Frame (帧) |
| **1. 物理层** | Physical Layer | 比特流在物理介质上的传输 | 电缆、光纤、无线电波 | Bit (比特) |

### 2.2 各层详细功能

#### 应用层 (Application Layer)
- 直接与用户的应用程序交互
- 提供文件传输（**FTP**）、网页浏览（**HTTP/HTTPS**）、电子邮件（**SMTP/POP3/IMAP**）等服务
- 数据以特定的应用协议格式编码

#### 传输层 (Transport Layer)
- 负责**端到端 (end-to-end)** 的数据传输
- **TCP (Transmission Control Protocol)**：可靠传输，有连接
- **UDP (User Datagram Protocol)**：不可靠但快速，无连接
- 使用 **port number（端口号）** 区分不同的应用

#### 网络层 (Network Layer)
- 负责 **logical addressing（逻辑寻址）**，即 IP 地址
- 负责 **routing（路由选择）**，决定数据包的传输路径
- 处理不同网络之间的通信

#### 数据链路层 (Data Link Layer)
- 负责 **physical addressing（物理寻址）**，即 MAC 地址
- 将数据封装成 **frame（帧）**
- 进行 **error detection（错误检测）** 和流量控制

#### 物理层 (Physical Layer)
- 负责在物理介质上传输 **raw bit stream（原始比特流）**
- 定义电压、频率、连接器类型等物理特性

In [ ]:
# ============================================================
# 可视化：TCP/IP 五层协议栈
# ============================================================

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('TCP/IP Protocol Stack（TCP/IP 协议栈）', fontsize=18, fontweight='bold', pad=20)

layers = [
    ('5. Application Layer\n应用层', '#FF6B6B', 'HTTP, FTP, SMTP, DNS', 'Data（数据）'),
    ('4. Transport Layer\n传输层', '#FFA07A', 'TCP, UDP', 'Segment（段）'),
    ('3. Network Layer\n网络层', '#FFD700', 'IP, ICMP, ARP', 'Packet（包）'),
    ('2. Data Link Layer\n数据链路层', '#98FB98', 'Ethernet, Wi-Fi', 'Frame（帧）'),
    ('1. Physical Layer\n物理层', '#87CEEB', 'Cable, Fiber, Radio', 'Bit（比特）'),
]

for i, (name, color, protocols, pdu) in enumerate(layers):
    y = 8.5 - i * 1.7
    # 主框
    rect = FancyBboxPatch((0.5, y - 0.6), 5, 1.2,
                          boxstyle="round,pad=0.1", facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(3, y, name, ha='center', va='center', fontsize=12, fontweight='bold')
    # 协议
    ax.text(7, y + 0.2, f'协议: {protocols}', ha='left', va='center', fontsize=10)
    ax.text(7, y - 0.2, f'PDU: {pdu}', ha='left', va='center', fontsize=10, style='italic')
    # 箭头
    if i < len(layers) - 1:
        ax.annotate('', xy=(3, y - 0.7), xytext=(3, y - 1.0),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

plt.tight_layout()
plt.show()

---

## 3. 协议封装 (Protocol Encapsulation)

当数据从应用层向下传递时，每一层都会添加自己的 **header（头部）**，这个过程叫做 **encapsulation（封装）**。

```
应用层:    [         DATA         ]
              ↓ 加上 TCP/UDP 头部
传输层:    [TCP Header][   DATA    ]   = Segment
              ↓ 加上 IP 头部
网络层:    [IP Header][TCP Header][DATA]  = Packet
              ↓ 加上帧头和帧尾
链路层:    [Frame Header][IP Header][TCP Header][DATA][Frame Trailer]  = Frame
              ↓ 转换为比特
物理层:    0110110101001010110010110...  = Bits
```

在接收端，每一层 **剥去 (strip)** 自己的头部，将剩余数据交给上一层，这叫做 **de-encapsulation（解封装）**。

> **考试重点**：封装过程是 Paper 3 的高频考点，务必记住每一层添加什么头部、数据单位叫什么名字。

In [ ]:
# ============================================================
# 可视化：协议封装过程 (Encapsulation Process)
# ============================================================

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Protocol Encapsulation（协议封装过程）', fontsize=18, fontweight='bold', pad=20)

# 定义每一层的封装块
encap_data = [
    # (y, blocks: [(x, width, label, color), ...], layer_label)
    (8.5, [(5, 4, 'DATA', '#FFB3B3')], 'Application Layer'),
    (6.8, [(3.5, 1.5, 'TCP\nHeader', '#FFD4A0'), (5, 4, 'DATA', '#FFB3B3')], 'Transport Layer → Segment'),
    (5.1, [(2, 1.5, 'IP\nHeader', '#FFFFA0'), (3.5, 1.5, 'TCP\nHeader', '#FFD4A0'), (5, 4, 'DATA', '#FFB3B3')], 'Network Layer → Packet'),
    (3.4, [(0.5, 1.5, 'Frame\nHeader', '#A0FFA0'), (2, 1.5, 'IP\nHeader', '#FFFFA0'), (3.5, 1.5, 'TCP\nHeader', '#FFD4A0'), (5, 4, 'DATA', '#FFB3B3'), (9, 1.5, 'Frame\nTrailer', '#A0FFA0')], 'Data Link Layer → Frame'),
    (1.7, [(0.5, 10, '0 1 1 0 1 1 0 1 0 1 0 0 1 0 1 0 1 1 0 0 1 0 1 1 0 ...', '#A0D4FF')], 'Physical Layer → Bits'),
]

for y, blocks, label in encap_data:
    for x, w, text, color in blocks:
        rect = FancyBboxPatch((x, y - 0.5), w, 1.0,
                              boxstyle="round,pad=0.05", facecolor=color, edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + w/2, y, text, ha='center', va='center', fontsize=9, fontweight='bold')
    ax.text(11.5, y, label, ha='left', va='center', fontsize=10, style='italic', color='#333333')
    if y > 2:
        ax.annotate('', xy=(5, y - 0.6), xytext=(5, y - 1.1),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

plt.tight_layout()
plt.show()

---

## 4. OSI 模型与 TCP/IP 模型的比较

**OSI (Open Systems Interconnection)** 模型是一个理论参考模型，共 **7层**。TCP/IP 是实际使用的模型，共 **4-5层**。

### 4.1 七层 OSI 模型

| OSI 层次 | 名称 | 对应 TCP/IP 层 | 功能说明 |
|---------|------|---------------|----------|
| 7 | **Application（应用层）** | Application | 用户接口，提供网络服务 |
| 6 | **Presentation（表示层）** | Application | 数据格式转换、加密、压缩 |
| 5 | **Session（会话层）** | Application | 管理会话的建立、维护和终止 |
| 4 | **Transport（传输层）** | Transport | 端到端可靠传输 |
| 3 | **Network（网络层）** | Network/Internet | 逻辑寻址和路由 |
| 2 | **Data Link（数据链路层）** | Data Link/Network Access | 物理寻址、帧处理 |
| 1 | **Physical（物理层）** | Data Link/Network Access | 比特流传输 |

### 4.2 关键区别

| 特征 | OSI 模型 | TCP/IP 模型 |
|------|---------|------------|
| 层数 | 7 层 | 4 层（或 5 层） |
| 性质 | **理论参考模型** | **实际使用模型** |
| 开发 | ISO 组织定义 | DARPA 项目开发 |
| 会话/表示层 | 单独分层 | 合并到应用层 |
| 协议定义 | 先有模型，后定义协议 | 先有协议，后总结模型 |
| 使用情况 | 主要用于**教学** | 互联网**实际使用** |

> **考试技巧**：记忆 OSI 七层可以用助记法 —— **A**ll **P**eople **S**eem **T**o **N**eed **D**ata **P**rocessing（从上到下：Application, Presentation, Session, Transport, Network, Data Link, Physical）

In [ ]:
# ============================================================
# 可视化：OSI 模型 vs TCP/IP 模型 对比图
# ============================================================

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 11)
ax.axis('off')
ax.set_title('OSI Model vs TCP/IP Model', fontsize=18, fontweight='bold', pad=20)

# OSI 模型（左侧）
osi_layers = [
    ('7. Application\n应用层', '#FF6B6B'),
    ('6. Presentation\n表示层', '#FF9B6B'),
    ('5. Session\n会话层', '#FFCB6B'),
    ('4. Transport\n传输层', '#FFA07A'),
    ('3. Network\n网络层', '#FFD700'),
    ('2. Data Link\n数据链路层', '#98FB98'),
    ('1. Physical\n物理层', '#87CEEB'),
]

ax.text(2.5, 10.3, 'OSI Model (7 layers)', ha='center', fontsize=14, fontweight='bold')
for i, (name, color) in enumerate(osi_layers):
    y = 9.3 - i * 1.2
    rect = FancyBboxPatch((0.5, y - 0.45), 4, 0.9,
                          boxstyle="round,pad=0.08", facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(2.5, y, name, ha='center', va='center', fontsize=9, fontweight='bold')

# TCP/IP 模型（右侧）
tcpip_layers = [
    ('Application\n应用层', '#FF6B6B', 3.6),  # 合并了 OSI 5,6,7
    ('Transport\n传输层', '#FFA07A', 1.2),
    ('Network (Internet)\n网络层', '#FFD700', 1.2),
    ('Data Link\n数据链路层', '#98FB98', 1.2),
    ('Physical\n物理层', '#87CEEB', 1.2),
]

ax.text(10.5, 10.3, 'TCP/IP Model (5 layers)', ha='center', fontsize=14, fontweight='bold')
tcp_y = 9.3
for name, color, height in tcpip_layers:
    rect = FancyBboxPatch((8.5, tcp_y - height + 0.45), 4, height - 0.1,
                          boxstyle="round,pad=0.08", facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(10.5, tcp_y - height/2 + 0.45, name, ha='center', va='center', fontsize=10, fontweight='bold')
    tcp_y -= height

# 连接线（虚线）
connections = [(9.3, 9.3), (5.7, 5.7), (4.5, 4.5), (3.3, 3.3), (2.1, 2.1)]
for osi_y, tcp_y in connections:
    ax.plot([4.5, 8.5], [osi_y, tcp_y], 'k--', alpha=0.3, linewidth=1)

# 大括号注释
ax.annotate('', xy=(5.5, 9.75), xytext=(5.5, 6.45),
            arrowprops=dict(arrowstyle='-', color='red', lw=2))
ax.text(6.2, 8.1, 'OSI 5,6,7 →\nTCP/IP\nApplication', ha='center', va='center',
        fontsize=9, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

---

## 5. TCP vs UDP —— 详细比较

### 5.1 TCP (Transmission Control Protocol)

TCP 是一个 **connection-oriented（面向连接）** 的协议。在传输数据之前，必须先建立连接（**three-way handshake / 三次握手**）。

#### 三次握手过程：

```
Client                          Server
  |                               |
  |--- SYN (seq=x) ------------->|    第1次：客户端发送 SYN
  |                               |
  |<-- SYN-ACK (seq=y, ack=x+1) -|    第2次：服务器回复 SYN-ACK
  |                               |
  |--- ACK (ack=y+1) ----------->|    第3次：客户端发送 ACK
  |                               |
  |====== 连接建立，开始传输 ======|
```

**TCP 的关键特性**：
- **Reliable（可靠）**：数据丢失会自动重传
- **Ordered（有序）**：数据包按正确顺序重组
- **Error-checked（错误检查）**：使用 checksum 检测错误
- **Flow control（流量控制）**：防止发送方过快
- **Congestion control（拥塞控制）**：避免网络过载

### 5.2 UDP (User Datagram Protocol)

UDP 是一个 **connectionless（无连接）** 的协议。不需要建立连接，直接发送数据。

```
Client                          Server
  |                               |
  |--- Data Packet 1 ----------->|    直接发送
  |--- Data Packet 2 ----------->|    不等确认
  |--- Data Packet 3 ----------->|    继续发送
  |                               |
```

**UDP 的关键特性**：
- **Fast（快速）**：没有握手开销
- **No guarantee（不保证）**：可能丢包、乱序
- **Lightweight（轻量级）**：头部只有 8 字节（TCP 为 20 字节）

### 5.3 完整比较表

| 特征 | TCP | UDP |
|------|-----|-----|
| 连接类型 | **Connection-oriented** | **Connectionless** |
| 可靠性 | 可靠（重传机制） | 不可靠 |
| 顺序保证 | 保证顺序 | 不保证 |
| 速度 | 较慢 | **较快** |
| 头部大小 | 20 bytes | **8 bytes** |
| 流量控制 | 有 | 无 |
| 拥塞控制 | 有 | 无 |
| 典型应用 | Web (HTTP), Email, File Transfer | DNS, Video Streaming, VoIP, Gaming |
| 适用场景 | 需要**准确性**的场景 | 需要**速度**的场景 |

In [ ]:
# ============================================================
# 可视化：TCP 三次握手 (Three-Way Handshake)
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# === 左图：TCP 三次握手 ===
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')
ax1.set_title('TCP Three-Way Handshake\nTCP 三次握手', fontsize=14, fontweight='bold')

# 客户端和服务器
ax1.text(2, 9.5, 'Client\n客户端', ha='center', fontsize=13, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#AED6F1', edgecolor='black'))
ax1.text(8, 9.5, 'Server\n服务器', ha='center', fontsize=13, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#ABEBC6', edgecolor='black'))

# 时间线
ax1.plot([2, 2], [8.5, 1], 'b-', linewidth=2)
ax1.plot([8, 8], [8.5, 1], 'g-', linewidth=2)

# 三次握手的箭头和标签
handshake = [
    (7.5, 'SYN (seq=100)', '#E74C3C', 2.2, 7.8),
    (5.5, 'SYN-ACK (seq=300, ack=101)', '#F39C12', 7.8, 2.2),
    (3.5, 'ACK (ack=301)', '#27AE60', 2.2, 7.8),
]

for y, label, color, x_from, x_to in handshake:
    ax1.annotate('', xy=(x_to, y - 0.5), xytext=(x_from, y),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax1.text(5, y + 0.1, label, ha='center', fontsize=10, fontweight='bold', color=color)

ax1.text(5, 1.8, '--- Connection Established ---', ha='center',
         fontsize=11, color='green', fontweight='bold', style='italic')

# === 右图：TCP vs UDP 对比 ===
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('TCP vs UDP Comparison', fontsize=14, fontweight='bold')

# TCP 侧
ax2.text(2.5, 9, 'TCP', ha='center', fontsize=16, fontweight='bold', color='#2E86C1')
tcp_features = [
    'Connection-oriented', 'Reliable', 'Ordered delivery',
    'Flow control', 'Error recovery', 'Slower', 'Header: 20 bytes'
]
for i, feat in enumerate(tcp_features):
    ax2.text(2.5, 8 - i * 0.9, f'• {feat}', ha='center', fontsize=10)

# 分隔线
ax2.plot([5, 5], [9.5, 1], 'k--', alpha=0.3)
ax2.text(5, 0.5, 'VS', ha='center', fontsize=14, fontweight='bold', color='red')

# UDP 侧
ax2.text(7.5, 9, 'UDP', ha='center', fontsize=16, fontweight='bold', color='#E74C3C')
udp_features = [
    'Connectionless', 'Unreliable', 'No ordering',
    'No flow control', 'No error recovery', 'Faster', 'Header: 8 bytes'
]
for i, feat in enumerate(udp_features):
    ax2.text(7.5, 8 - i * 0.9, f'• {feat}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 演示：TCP 和 UDP 报文头部结构
# ============================================================

print("=" * 70)
print("TCP Header Structure (TCP 报文头部结构)")
print("=" * 70)
print("""
 0                   1                   2                   3
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|         Source Port (16)      |      Destination Port (16)    |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                    Sequence Number (32)                       |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                 Acknowledgment Number (32)                    |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|Offset| Res |U|A|P|R|S|F|          Window (16)                |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|         Checksum (16)         |       Urgent Pointer (16)     |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                    Options (variable)                         |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

  最小头部大小: 20 bytes (没有 Options 时)
  标志位 (Flags): U=URG, A=ACK, P=PSH, R=RST, S=SYN, F=FIN
""")

print("=" * 70)
print("UDP Header Structure (UDP 报文头部结构)")
print("=" * 70)
print("""
 0                   1                   2                   3
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|         Source Port (16)      |      Destination Port (16)    |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|            Length (16)        |          Checksum (16)        |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

  头部大小: 仅 8 bytes！比 TCP 简单得多！
  只有 4 个字段，没有序列号、确认号、标志位等
""")

print("💡 对比: TCP header = 20+ bytes  vs  UDP header = 8 bytes")
print("   UDP 更轻量，因此在实时通信场景中更受欢迎。")

---

## 6. IP 地址 (IP Addressing)

### 6.1 IPv4 地址

**IPv4 (Internet Protocol version 4)** 使用 **32 位** 地址，通常以 **点分十进制 (dotted decimal)** 表示。

```
二进制:    11000000.10101000.00000001.01100100
十进制:    192     .168     .1       .100
```

**IPv4 地址范围**：`0.0.0.0` 到 `255.255.255.255`，共约 **43亿** (2^32 = 4,294,967,296) 个地址。

#### IPv4 地址分类（传统分类方式）：

| 类别 | 第一字节范围 | 默认子网掩码 | 用途 |
|------|------------|-------------|------|
| **Class A** | 1-126 | 255.0.0.0 (/8) | 大型网络 |
| **Class B** | 128-191 | 255.255.0.0 (/16) | 中型网络 |
| **Class C** | 192-223 | 255.255.255.0 (/24) | 小型网络 |
| **Class D** | 224-239 | — | 多播 (Multicast) |
| **Class E** | 240-255 | — | 保留 (Reserved) |

#### 私有 IP 地址范围（不在互联网上路由）：
- **10.0.0.0** – 10.255.255.255（Class A）
- **172.16.0.0** – 172.31.255.255（Class B）
- **192.168.0.0** – 192.168.255.255（Class C）

### 6.2 IPv6 地址

由于 IPv4 地址即将耗尽，**IPv6** 使用 **128 位** 地址，以 **冒号十六进制 (colon hexadecimal)** 表示。

```
完整形式:  2001:0db8:85a3:0000:0000:8a2e:0370:7334
缩写形式:  2001:db8:85a3::8a2e:370:7334
```

**IPv6 的优势**：
- 地址空间极大：2^128 ≈ 3.4 × 10^38 个地址
- 内置安全性（IPsec）
- 简化的报文头部
- 更好的多播支持
- 无需 NAT

In [ ]:
# ============================================================
# 演示：IPv4 地址解析和分析
# ============================================================
import ipaddress

def analyze_ipv4(ip_str):
    """详细分析一个 IPv4 地址"""
    ip = ipaddress.IPv4Address(ip_str)
    
    # 转换为二进制
    octets = ip_str.split('.')
    binary_octets = [format(int(o), '08b') for o in octets]
    binary_str = '.'.join(binary_octets)
    
    print(f"{'='*60}")
    print(f"IP 地址分析: {ip_str}")
    print(f"{'='*60}")
    print(f"十进制表示:    {ip_str}")
    print(f"二进制表示:    {binary_str}")
    print(f"整数值:        {int(ip)}")
    print(f"是否私有地址:  {'是 (Private)' if ip.is_private else '否 (Public)'}")
    print(f"是否回环地址:  {'是 (Loopback)' if ip.is_loopback else '否'}")
    print(f"是否多播地址:  {'是 (Multicast)' if ip.is_multicast else '否'}")
    print()

# 分析几个示例 IP 地址
test_ips = ['192.168.1.100', '10.0.0.1', '8.8.8.8', '127.0.0.1', '172.16.0.1']
for ip in test_ips:
    analyze_ipv4(ip)

In [ ]:
# ============================================================
# 演示：IPv4 vs IPv6 对比
# ============================================================

print("=" * 65)
print("IPv4 vs IPv6 Comparison（IPv4 与 IPv6 对比）")
print("=" * 65)

comparison = [
    ("地址长度", "32 bits", "128 bits"),
    ("地址数量", f"2^32 = {2**32:,}", f"2^128 ≈ 3.4 × 10^38"),
    ("表示方式", "点分十进制 (192.168.1.1)", "冒号十六进制 (2001:db8::1)"),
    ("分隔符", "点 (.)", "冒号 (:)"),
    ("地址示例", "192.168.1.100", "2001:0db8:85a3::8a2e:0370:7334"),
    ("NAT 需求", "需要 NAT", "不需要 NAT"),
    ("安全性", "IPsec 可选", "IPsec 内置"),
    ("报文头部", "可变长度（20-60 bytes）", "固定长度（40 bytes）"),
    ("广播", "支持 broadcast", "使用 multicast 替代"),
]

print(f"{'特征':<15} {'IPv4':<30} {'IPv6'}")
print("-" * 80)
for feature, v4, v6 in comparison:
    print(f"{feature:<15} {v4:<30} {v6}")

print("\n" + "=" * 65)
print("IPv6 地址缩写规则:")
print("=" * 65)

ipv6_full = "2001:0db8:85a3:0000:0000:8a2e:0370:7334"
ipv6_obj = ipaddress.IPv6Address(ipv6_full)
print(f"完整形式 (Expanded):  {ipv6_full}")
print(f"规则1 - 去掉前导零:  2001:db8:85a3:0:0:8a2e:370:7334")
print(f"规则2 - 连续的0用::代替: {ipv6_obj.compressed}")
print(f"\n注意: :: 在一个地址中只能使用一次！")

---

## 7. 子网划分与 CIDR (Subnetting & CIDR Notation)

### 7.1 Subnet Mask（子网掩码）

**子网掩码 (Subnet Mask)** 用于区分 IP 地址中的 **network part（网络部分）** 和 **host part（主机部分）**。

```
IP 地址:     192.168.1.100
             11000000.10101000.00000001.01100100

子网掩码:    255.255.255.0
             11111111.11111111.11111111.00000000
             |-------- 网络部分 --------| 主机部分 |

网络地址:    192.168.1.0    (IP AND 子网掩码)
主机部分:    0.0.0.100
```

### 7.2 CIDR 表示法

**CIDR (Classless Inter-Domain Routing)** 使用斜线符号表示网络前缀长度：

- `192.168.1.0/24` 表示前 24 位是网络部分
- 等价于子网掩码 `255.255.255.0`

常见 CIDR 前缀：

| CIDR | 子网掩码 | 可用主机数 | 说明 |
|------|---------|-----------|------|
| /8 | 255.0.0.0 | 16,777,214 | Class A |
| /16 | 255.255.0.0 | 65,534 | Class B |
| /24 | 255.255.255.0 | 254 | Class C |
| /25 | 255.255.255.128 | 126 | 半个 Class C |
| /26 | 255.255.255.192 | 62 | 1/4 Class C |
| /28 | 255.255.255.240 | 14 | 1/16 Class C |
| /30 | 255.255.255.252 | 2 | 点对点链路 |
| /32 | 255.255.255.255 | 1 | 单个主机 |

> **公式**：可用主机数 = **2^(32 - prefix) - 2**（减去网络地址和广播地址）

In [ ]:
# ============================================================
# 演示：子网划分计算器 (Subnetting Calculator)
# ============================================================

def subnet_calculator(cidr_str):
    """子网划分计算器：输入 CIDR 格式，输出详细信息"""
    network = ipaddress.IPv4Network(cidr_str, strict=False)
    
    # 获取各项信息
    ip_part = cidr_str.split('/')[0]
    prefix_len = network.prefixlen
    
    # 二进制表示
    ip_obj = ipaddress.IPv4Address(ip_part)
    ip_binary = format(int(ip_obj), '032b')
    mask_binary = format(int(network.netmask), '032b')
    
    # 格式化二进制（每8位一组）
    ip_bin_fmt = '.'.join([ip_binary[i:i+8] for i in range(0, 32, 8)])
    mask_bin_fmt = '.'.join([mask_binary[i:i+8] for i in range(0, 32, 8)])
    
    print(f"{'='*65}")
    print(f"子网划分分析: {cidr_str}")
    print(f"{'='*65}")
    print(f"IP 地址:         {ip_part}")
    print(f"IP (二进制):     {ip_bin_fmt}")
    print(f"子网掩码:        {network.netmask}")
    print(f"掩码 (二进制):   {mask_bin_fmt}")
    print(f"CIDR 前缀长度:   /{prefix_len}")
    print(f"网络地址:        {network.network_address}")
    print(f"广播地址:        {network.broadcast_address}")
    print(f"第一个可用地址:  {list(network.hosts())[0] if network.num_addresses > 2 else 'N/A'}")
    print(f"最后一个可用地址: {list(network.hosts())[-1] if network.num_addresses > 2 else 'N/A'}")
    print(f"可用主机数:      {network.num_addresses - 2}")
    print(f"总地址数:        {network.num_addresses}")
    print()
    
    # 可视化网络位和主机位
    net_bits = ip_binary[:prefix_len]
    host_bits = ip_binary[prefix_len:]
    print(f"网络部分 ({prefix_len} bits):  {net_bits}")
    print(f"主机部分 ({32-prefix_len} bits): {'.' * prefix_len}{host_bits}")
    print()

# 测试不同的子网
test_subnets = [
    '192.168.1.100/24',
    '10.0.0.50/8',
    '172.16.0.1/25',
    '192.168.10.0/28',
]

for subnet in test_subnets:
    subnet_calculator(subnet)

In [ ]:
# ============================================================
# 可视化：子网掩码与网络/主机位划分
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 8))
fig.suptitle('Subnetting Visualization（子网划分可视化）', fontsize=16, fontweight='bold')

examples = [
    ('192.168.1.100/24', 24),
    ('10.50.30.200/16', 16),
    ('172.16.0.50/25', 25),
]

for idx, (cidr, prefix) in enumerate(examples):
    ax = axes[idx]
    ip_part = cidr.split('/')[0]
    ip_int = int(ipaddress.IPv4Address(ip_part))
    ip_binary = format(ip_int, '032b')
    
    ax.set_xlim(0, 33)
    ax.set_ylim(0, 2)
    ax.axis('off')
    ax.set_title(f'{cidr}', fontsize=12, fontweight='bold', loc='left')
    
    for i in range(32):
        color = '#FF9999' if i < prefix else '#99CCFF'  # 红=网络位, 蓝=主机位
        rect = plt.Rectangle((i + 0.5, 0.6), 0.9, 0.8, facecolor=color, edgecolor='black', linewidth=0.5)
        ax.add_patch(rect)
        ax.text(i + 0.95, 1.0, ip_binary[i], ha='center', va='center', fontsize=8, fontweight='bold')
        # 八位组分隔
        if i > 0 and i % 8 == 0:
            ax.plot([i + 0.5, i + 0.5], [0.5, 1.5], 'k-', linewidth=2)
    
    # 网络前缀分隔线
    ax.plot([prefix + 0.5, prefix + 0.5], [0.3, 1.7], 'r-', linewidth=3)
    ax.text(prefix / 2 + 0.5, 0.3, f'Network ({prefix} bits)', ha='center', fontsize=9, color='red')
    ax.text(prefix + (32 - prefix) / 2 + 0.5, 0.3, f'Host ({32-prefix} bits)', ha='center', fontsize=9, color='blue')

# 图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#FF9999', edgecolor='black', label='Network bits（网络位）'),
    Patch(facecolor='#99CCFF', edgecolor='black', label='Host bits（主机位）'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=11)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

---

## 8. 端口号与套接字 (Port Numbers & Sockets)

### 8.1 Port Numbers（端口号）

**端口号** 是一个 16 位整数（范围 **0-65535**），用于标识一台计算机上的特定应用或服务。

#### 端口号分类：

| 范围 | 名称 | 说明 |
|------|------|------|
| **0 - 1023** | Well-known ports（知名端口） | 分配给常用服务 |
| **1024 - 49151** | Registered ports（注册端口） | 分配给特定应用 |
| **49152 - 65535** | Dynamic/Private ports（动态/私有端口） | 临时使用 |

#### 常见知名端口（考试必须记住！）：

| 端口号 | 协议 | 服务 |
|--------|------|------|
| **20, 21** | FTP | 文件传输（20=数据, 21=控制） |
| **22** | SSH | 安全远程登录 |
| **23** | Telnet | 远程登录（不安全） |
| **25** | SMTP | 发送电子邮件 |
| **53** | DNS | 域名解析 |
| **80** | HTTP | 网页浏览 |
| **110** | POP3 | 接收电子邮件 |
| **143** | IMAP | 接收电子邮件（高级） |
| **443** | HTTPS | 安全网页浏览 |

### 8.2 Socket（套接字）

**Socket = IP Address + Port Number**

套接字是网络通信的端点。一个完整的 TCP 连接由一对套接字标识：

```
客户端套接字:  192.168.1.100:52431  (IP:端口)
服务器套接字:  93.184.216.34:443    (IP:端口)

连接标识:  (192.168.1.100, 52431, 93.184.216.34, 443)
```

> **考试重点**：Socket 是一个**端点 (endpoint)**，由 IP 地址和端口号的组合唯一标识。

In [ ]:
# ============================================================
# 演示：常见端口号查询表
# ============================================================

well_known_ports = {
    20: ('FTP Data', 'TCP', '文件传输 - 数据通道'),
    21: ('FTP Control', 'TCP', '文件传输 - 控制通道'),
    22: ('SSH', 'TCP', '安全远程登录'),
    23: ('Telnet', 'TCP', '远程登录（不安全）'),
    25: ('SMTP', 'TCP', '发送电子邮件'),
    53: ('DNS', 'TCP/UDP', '域名系统'),
    67: ('DHCP Server', 'UDP', '动态主机配置（服务器）'),
    68: ('DHCP Client', 'UDP', '动态主机配置（客户端）'),
    80: ('HTTP', 'TCP', '超文本传输协议'),
    110: ('POP3', 'TCP', '邮件接收（下载）'),
    143: ('IMAP', 'TCP', '邮件接收（同步）'),
    443: ('HTTPS', 'TCP', '安全超文本传输协议'),
}

print(f"{'='*70}")
print(f"Well-Known Ports（常见端口号）- 考试必备！")
print(f"{'='*70}")
print(f"{'端口':<8} {'服务':<16} {'协议':<10} {'说明'}")
print(f"{'-'*70}")
for port, (service, proto, desc) in sorted(well_known_ports.items()):
    print(f"{port:<8} {service:<16} {proto:<10} {desc}")

In [ ]:
# ============================================================
# 演示：Socket 编程基础 - 模拟客户端/服务器通信概念
# ============================================================

print("=" * 60)
print("Socket 编程基础概念演示")
print("=" * 60)
print()

# 展示 socket 地址的构成
class SocketDemo:
    """模拟 Socket 通信概念"""
    def __init__(self, ip, port, role):
        self.ip = ip
        self.port = port
        self.role = role
    
    def __str__(self):
        return f"{self.ip}:{self.port}"
    
    def describe(self):
        print(f"  {self.role}:")
        print(f"    IP Address:  {self.ip}")
        print(f"    Port Number: {self.port}")
        print(f"    Socket:      {self}")

# 模拟一次 HTTPS 连接
print("模拟场景: 浏览器访问 www.example.com")
print("-" * 50)

client = SocketDemo('192.168.1.100', 52431, 'Client（客户端/浏览器）')
server = SocketDemo('93.184.216.34', 443, 'Server（服务器）')

client.describe()
print()
server.describe()

print(f"\n完整连接标识 (Connection Tuple):")
print(f"  ({client.ip}, {client.port}, {server.ip}, {server.port})")
print(f"\n这个四元组唯一标识了这一个 TCP 连接。")
print(f"同一台电脑可以同时打开多个网页，因为每个连接使用不同的客户端端口号！")

print("\n" + "=" * 60)
print("多个同时连接的示例:")
print("=" * 60)
connections = [
    ('192.168.1.100', 52431, '93.184.216.34', 443, 'Tab 1: www.example.com'),
    ('192.168.1.100', 52432, '142.250.80.46', 443, 'Tab 2: www.google.com'),
    ('192.168.1.100', 52433, '93.184.216.34', 443, 'Tab 3: www.example.com (另一个标签页)'),
]

for src_ip, src_port, dst_ip, dst_port, desc in connections:
    print(f"  {desc}")
    print(f"    {src_ip}:{src_port} ←→ {dst_ip}:{dst_port}")

In [ ]:
# ============================================================
# 可视化：IP Packet 结构
# ============================================================

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 32)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('IPv4 Packet Header Structure（IPv4 数据包头部结构）', fontsize=16, fontweight='bold', pad=15)

# 位标尺
for i in range(0, 33, 4):
    ax.text(i, 11.2, str(i), ha='center', fontsize=8, color='gray')

# IPv4 头部字段
fields = [
    # (x, y, width, height, label, color)
    (0, 10, 4, 0.9, 'Version\n(4)', '#FFB3B3'),
    (4, 10, 4, 0.9, 'IHL\n(4)', '#FFD4B3'),
    (8, 10, 8, 0.9, 'Type of Service\n(8)', '#FFFFB3'),
    (16, 10, 16, 0.9, 'Total Length\n(16)', '#B3FFB3'),
    
    (0, 8.8, 16, 0.9, 'Identification\n(16)', '#B3FFFF'),
    (16, 8.8, 3, 0.9, 'Flags\n(3)', '#B3B3FF'),
    (19, 8.8, 13, 0.9, 'Fragment Offset\n(13)', '#FFB3FF'),
    
    (0, 7.6, 8, 0.9, 'TTL\n(8)', '#FFD4B3'),
    (8, 7.6, 8, 0.9, 'Protocol\n(8)', '#B3FFD4'),
    (16, 7.6, 16, 0.9, 'Header Checksum\n(16)', '#D4B3FF'),
    
    (0, 6.4, 32, 0.9, 'Source IP Address (32 bits)', '#FFCCCB'),
    (0, 5.2, 32, 0.9, 'Destination IP Address (32 bits)', '#C1E1C1'),
    (0, 4.0, 32, 0.9, 'Options (variable) + Padding', '#E8E8E8'),
    (0, 2.5, 32, 1.2, 'DATA (Payload / 数据负载)', '#FFFACD'),
]

for x, y, w, h, label, color in fields:
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='black', linewidth=1)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=8, fontweight='bold')

# 注释
ax.text(16, 1.3, 'TTL = Time To Live（生存时间，每经过一个路由器减 1，减到 0 则丢弃）',
        ha='center', fontsize=9, style='italic')
ax.text(16, 0.7, 'Protocol: 6=TCP, 17=UDP, 1=ICMP',
        ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 演示：模拟数据包封装过程 (Packet Encapsulation Simulation)
# ============================================================

def simulate_encapsulation(message, src_ip, dst_ip, src_port, dst_port, protocol='TCP'):
    """模拟数据从应用层到物理层的封装过程"""
    
    print(f"{'='*70}")
    print(f"模拟封装过程: 从 {src_ip}:{src_port} 发送到 {dst_ip}:{dst_port}")
    print(f"消息内容: \"{message}\"")
    print(f"{'='*70}\n")
    
    # 应用层
    app_data = message.encode('utf-8')
    print(f"Layer 5 - Application Layer（应用层）:")
    print(f"  数据: {message}")
    print(f"  编码后: {app_data}")
    print(f"  大小: {len(app_data)} bytes\n")
    
    # 传输层 - 添加 TCP/UDP 头部
    print(f"Layer 4 - Transport Layer（传输层）:")
    print(f"  协议: {protocol}")
    print(f"  源端口 (Source Port): {src_port}")
    print(f"  目标端口 (Dest Port): {dst_port}")
    tcp_header_size = 20 if protocol == 'TCP' else 8
    segment_size = tcp_header_size + len(app_data)
    print(f"  {protocol} Header: {tcp_header_size} bytes")
    print(f"  Segment 总大小: {segment_size} bytes")
    print(f"  结构: [{protocol} Header ({tcp_header_size}B)][Data ({len(app_data)}B)]\n")
    
    # 网络层 - 添加 IP 头部
    print(f"Layer 3 - Network Layer（网络层）:")
    print(f"  源 IP (Source IP): {src_ip}")
    print(f"  目标 IP (Dest IP): {dst_ip}")
    ip_header_size = 20
    packet_size = ip_header_size + segment_size
    print(f"  IP Header: {ip_header_size} bytes")
    print(f"  Packet 总大小: {packet_size} bytes")
    print(f"  结构: [IP Header ({ip_header_size}B)][{protocol} Header ({tcp_header_size}B)][Data ({len(app_data)}B)]\n")
    
    # 数据链路层 - 添加帧头和帧尾
    print(f"Layer 2 - Data Link Layer（数据链路层）:")
    frame_header = 14  # Ethernet header
    frame_trailer = 4  # CRC/FCS
    frame_size = frame_header + packet_size + frame_trailer
    print(f"  Frame Header (Ethernet): {frame_header} bytes")
    print(f"  Frame Trailer (CRC): {frame_trailer} bytes")
    print(f"  Frame 总大小: {frame_size} bytes")
    print(f"  结构: [Eth Header ({frame_header}B)][IP ({ip_header_size}B)][{protocol} ({tcp_header_size}B)][Data ({len(app_data)}B)][CRC ({frame_trailer}B)]\n")
    
    # 物理层
    total_bits = frame_size * 8
    print(f"Layer 1 - Physical Layer（物理层）:")
    print(f"  转换为比特流: {total_bits} bits")
    print(f"  在物理介质上以电信号/光信号/无线信号传输\n")
    
    print(f"{'='*70}")
    print(f"总结: 原始数据 {len(app_data)} bytes → 最终帧 {frame_size} bytes")
    print(f"协议开销 (Overhead): {frame_size - len(app_data)} bytes ({(frame_size - len(app_data))/frame_size*100:.1f}%)")

# 运行模拟
simulate_encapsulation(
    message="GET / HTTP/1.1\r\nHost: www.example.com",
    src_ip="192.168.1.100",
    dst_ip="93.184.216.34",
    src_port=52431,
    dst_port=80,
    protocol='TCP'
)

In [ ]:
# ============================================================
# Socket 编程入门示例：使用 Python 获取网站 IP
# ============================================================

import socket

print("=" * 60)
print("Socket 编程入门 - DNS 域名解析")
print("=" * 60)
print()

# 域名解析
domains = ['www.google.com', 'www.baidu.com', 'www.github.com', 'www.python.org']

for domain in domains:
    try:
        ip = socket.gethostbyname(domain)
        print(f"  {domain:<25} → {ip}")
    except socket.gaierror:
        print(f"  {domain:<25} → [解析失败]")

print("\n" + "=" * 60)
print("获取本机网络信息")
print("=" * 60)
print()

hostname = socket.gethostname()
print(f"  主机名 (Hostname):    {hostname}")
try:
    local_ip = socket.gethostbyname(hostname)
    print(f"  本机 IP 地址:          {local_ip}")
except:
    print(f"  本机 IP 地址:          [无法获取]")

print("\n" + "=" * 60)
print("Socket 类型说明")
print("=" * 60)
print(f"  socket.SOCK_STREAM = {socket.SOCK_STREAM}  → TCP (面向连接的流式套接字)")
print(f"  socket.SOCK_DGRAM  = {socket.SOCK_DGRAM}  → UDP (无连接的数据报套接字)")
print(f"  socket.AF_INET     = {socket.AF_INET}  → IPv4 地址族")
print(f"  socket.AF_INET6    = {socket.AF_INET6} → IPv6 地址族")

---

## 9. 综合回顾 (Summary)

### 考试要点清单

| 考点 | 关键知识 |
|------|----------|
| Protocol Stack | 分层架构，每层有特定功能和 PDU |
| TCP/IP 五层 | Application → Transport → Network → Data Link → Physical |
| OSI 七层 | 比 TCP/IP 多了 Presentation 和 Session 层 |
| Encapsulation | 数据向下传递时，每层添加头部 |
| TCP vs UDP | TCP=可靠/有连接；UDP=快速/无连接 |
| Three-way handshake | SYN → SYN-ACK → ACK |
| IPv4 | 32 位，点分十进制，约 43 亿地址 |
| IPv6 | 128 位，冒号十六进制，地址几乎无限 |
| Subnetting | 子网掩码区分网络位和主机位 |
| CIDR | /24 表示前 24 位是网络部分 |
| Port numbers | 0-1023 = well-known, 16 位整数 |
| Socket | IP + Port，网络通信的端点 |

---

## 10. 练习题 (Practice Exercises)

### 练习 1：基础概念题

**Q1.** TCP/IP 协议栈共有几层？请从上到下列出每一层的名称和对应的 PDU。

**Q2.** 解释 "encapsulation（封装）" 的概念，并描述数据从应用层到物理层的封装过程。

**Q3.** 比较 TCP 和 UDP 的 **三个** 主要区别，并各举一个适合使用的应用场景。

**Q4.** 什么是 "three-way handshake（三次握手）"？请画出过程图并解释每一步。

---

### 练习 2：IP 地址计算

**Q5.** 将 IP 地址 `172.16.254.1` 转换为 32 位二进制表示。

**Q6.** 给定网络 `192.168.10.0/26`，计算：
- (a) 子网掩码（十进制）
- (b) 网络地址
- (c) 广播地址
- (d) 可用主机数
- (e) 第一个和最后一个可用 IP

**Q7.** 一个组织需要为 50 台设备分配 IP 地址。最小的 CIDR 前缀长度应该是多少？为什么？

---

### 练习 3：编程挑战

In [ ]:
# ============================================================
# 练习 3a：编写一个函数，判断给定 IP 地址的传统类别
# ============================================================

def classify_ip(ip_str):
    """
    判断一个 IPv4 地址属于 Class A/B/C/D/E
    
    参数:
        ip_str: 字符串格式的 IPv4 地址 (如 "192.168.1.1")
    
    返回:
        字符串，表示地址类别 (如 "Class C")
    
    提示:
        - Class A: 第一字节 1-126
        - Class B: 第一字节 128-191
        - Class C: 第一字节 192-223
        - Class D: 第一字节 224-239
        - Class E: 第一字节 240-255
    """
    # 在下面写你的代码
    # first_octet = ...
    # if ...:
    #     return "Class A"
    # ...
    pass  # 删除这行，写你的代码

# 测试
test_ips = ['10.0.0.1', '172.16.0.1', '192.168.1.1', '224.0.0.1', '240.0.0.1']
# 预期输出: Class A, Class B, Class C, Class D, Class E
for ip in test_ips:
    result = classify_ip(ip)
    print(f"{ip} → {result}")

In [ ]:
# ============================================================
# 练习 3b：编写子网计算函数
# ============================================================

def calculate_subnet_info(ip_str, prefix_length):
    """
    给定一个 IP 地址和 CIDR 前缀长度，计算子网信息。
    
    不要使用 ipaddress 库！用基本的位运算来实现。
    
    参数:
        ip_str: 字符串格式的 IPv4 地址
        prefix_length: CIDR 前缀长度 (0-32 的整数)
    
    返回:
        一个字典，包含:
        - 'subnet_mask': 子网掩码字符串
        - 'network_address': 网络地址字符串
        - 'broadcast_address': 广播地址字符串
        - 'usable_hosts': 可用主机数 (整数)
    
    提示:
        1. 将 IP 转为 32 位整数
        2. 子网掩码 = 前 prefix_length 位全为 1，其余为 0
        3. 网络地址 = IP AND 子网掩码
        4. 广播地址 = 网络地址 OR (NOT 子网掩码)
    """
    # 在下面写你的代码
    pass

# 测试
# result = calculate_subnet_info('192.168.1.100', 24)
# print(result)
# 预期: {'subnet_mask': '255.255.255.0', 'network_address': '192.168.1.0',
#         'broadcast_address': '192.168.1.255', 'usable_hosts': 254}

---

### 练习 4：Past Paper 风格题

**Q8.** (6 marks) A company uses TCP/IP to transmit data across its network.

(a) Describe the role of **each layer** in the TCP/IP protocol stack when a user sends an email. [5]

(b) Explain why a **layered approach** is beneficial for network communication. [1]

---

**Q9.** (4 marks) Compare TCP and UDP. State **two differences** and give **one example application** for each protocol.

---

**Q10.** (5 marks) A network has the address `192.168.5.0/26`.

(a) Calculate the subnet mask. [1]

(b) Calculate the number of usable host addresses. [1]

(c) State the range of usable host IP addresses. [2]

(d) Explain what happens when a packet's TTL reaches 0. [1]

---

> **下一节预告**：在 `02_互联网技术.ipynb` 中，我们将学习 DNS 解析、HTTP/HTTPS 协议、SSL/TLS 握手、Web 技术和电子邮件协议等内容。